Interactive Demo for AnimeGANv2:FacePortraitv2 created by @xhlulu

Learn more about the model here: https://github.com/bryandlee/animegan2-pytorch

To start using this, run the cells with `Ctrl+F9` or "Runtime > Run All"

For accelerated inference, you can use a GPU. Simply select "Runtime > Change runtime type" and select "GPU" in the "Hardware Acceleration" dropdown.


In [ ]:
#@title Anime FaceGAN Colab app# 这是一个Colab应用的标题，通常用于表单输入。from io import BytesIO # 用于在内存中处理二进制流，例如图片数据。import torch # PyTorch核心库。from PIL import Image # Pillow库，用于图像处理。import ipywidgets as widgets # 用于创建交互式UI组件，如按钮、输出区域等。import IPython.display as display # 用于在Jupyter环境中显示内容，包括widgets。from google.colab import files # Google Colab提供的文件处理工具，如此处的上传功能。# 检查是否有可用的CUDA GPU，否则使用CPU。device = "cuda" if torch.cuda.is_available() else "cpu"# 通过PyTorch Hub加载预训练的AnimeGANv2生成器模型。# "bryandlee/animegan2-pytorch:main" 指定了GitHub仓库和分支。# "generator" 是hubconf.py中定义的模型加载函数名。# device=device 指定模型加载到的设备。# .eval() 将模型设置为评估模式（禁用dropout等）。model = torch.hub.load("bryandlee/animegan2-pytorch:main", "generator", device=device).eval()# 通过PyTorch Hub加载face2paint辅助函数，用于简化图像转换流程。face2paint = torch.hub.load("bryandlee/animegan2-pytorch:main", "face2paint", device=device)# @param指令用于在Colab中创建表单，这里让用户选择输出图片的格式。image_format = "png" #@param ["jpeg", "png"]# 创建一个交互式按钮。button = widgets.Button(description="Start")# 创建一个输出区域，用于显示结果。output = widgets.Output()# 定义按钮点击时执行的回调函数。def run(b):    # 禁用按钮，防止重复点击。    button.disabled = True    # 清空上一次的输出内容。    with output:        display.clear_output()        # 调用Colab的文件上传接口，让用户上传图片。    uploaded = files.upload()    # 遍历所有上传的文件。    for fname in uploaded:        # 获取上传文件的二进制内容。        bytes_in = uploaded[fname]        # 从二进制内容中打开图片，并确保是RGB格式。        im_in = Image.open(BytesIO(bytes_in)).convert("RGB")        # 调用face2paint函数进行动漫风格转换。        # model: 加载的生成器模型。        # im_in: 输入的PIL Image对象。        # side_by_side=False: 不将原图和结果图并排显示（在此app中分别显示）。        im_out = face2paint(model, im_in, side_by_side=False)        # 创建一个内存中的二进制流，用于保存输出图片。        buffer_out = BytesIO()        # 将处理后的PIL Image对象保存到内存流中，格式由image_format变量指定。        im_out.save(buffer_out, format=image_format)        # 获取输出图片的二进制内容。        bytes_out = buffer_out.getvalue()        # 创建ipywidgets的Image组件来显示输入图片。        wi1 = widgets.Image(value=bytes_in, format=image_format)        # 创建ipywidgets的Image组件来显示输出图片。        wi2 = widgets.Image(value=bytes_out, format=image_format)        # 设置输入图片显示的最大宽度和高度。        wi1.layout.max_width = '500px'        wi1.layout.max_height = '500px'        # 设置输出图片显示的最大宽度和高度。        wi2.layout.max_width = '500px'        wi2.layout.max_height = '500px'        ## 使用HBox组件将输入和输出图片并排显示。        sidebyside = widgets.HBox([wi1, wi2])        ## 在输出区域显示并排的图片。        with output:            display.display(sidebyside)        # 处理完成后，重新启用按钮。    button.disabled = False# 将run函数注册为按钮的点击事件回调。button.on_click(run)# 在Colab中显示按钮和输出区域。display.display(button, output)